# Circuitree dataset: quick start (Colab)

Find circuits that compute the **same Boolean function** even when their gates are wired differently.
This notebook downloads the data, shows its format, and trains a small example model.
See `README.md` in the dataset folder for the full description.

## 1. Get the data
Set `DATA_URL` to the link your team lead shared (GitHub release or Google Drive).

In [ ]:
DATA_URL = "PASTE_THE_DATASET_LINK_HERE"   # e.g. https://github.com/<org>/<repo>/releases/download/v2.0/circuitree_v2.zip

import os
if not os.path.exists("circuitree_v2"):
    if "drive.google.com" in DATA_URL:
        !pip -q install gdown
        !gdown --fuzzy "$DATA_URL" -O circuitree_v2.zip
    else:
        !wget -q "$DATA_URL" -O circuitree_v2.zip
    !unzip -q circuitree_v2.zip
%cd circuitree_v2
!ls

## 2. Look at the data
Each sample is a `.bench` file. All labels are in `samples.csv`.

In [ ]:
import circuitree_data as cd

train = cd.read_samples(track="synthetic", split="train", role="view")
print(len(train), "training views; example row:")
print(train[0])
print(open(train[0]["file"]).read())

In [ ]:
# Two views of the same function: different wiring, identical behaviour.
from collections import defaultdict
by_fn = defaultdict(list)
for r in train:
    by_fn[r["function_id"]].append(r)
fid, views = next((f, v) for f, v in by_fn.items() if len(v) >= 2)
a, b = cd.parse_bench(views[0]["file"]), cd.parse_bench(views[1]["file"])
k = len(a["inputs"])
same = all(cd.simulate(a, [(t >> i) & 1 for i in range(k)]) == cd.simulate(b, [(t >> i) & 1 for i in range(k)])
           for t in range(2 ** k))
print(fid, "views:", len(views), "| AND gates:", views[0]["and_count"], "vs", views[1]["and_count"], "| same truth table:", same)

In [ ]:
# Graph form of one circuit (networkx); inverters are edge attributes.
g = cd.bench_to_networkx(views[0]["file"])
print(g.number_of_nodes(), "nodes,", g.number_of_edges(), "edges")

## 3. Train the example model (optional)
Needs PyTorch Geometric. On a CPU runtime the synthetic track takes a few minutes.

In [ ]:
!pip -q install torch_geometric
!python example_train.py --track synthetic --steps 500 --near-miss

## 4. Evaluate your own model
Produce one vector per sample of the split and pass them to `evaluate.py`.
It does not matter how you built your graphs or model.

In [ ]:
import numpy as np, evaluate as ev
test = cd.read_samples(track="synthetic", split="test")
my_embeddings = {r["sample_id"]: np.random.randn(32) for r in test}   # <- replace with your model's output
print(ev.evaluate(my_embeddings, "synthetic", "test", "hard"))
print(ev.evaluate(my_embeddings, "synthetic", "test", "full"))